# PA2: Convolutional Neural Networks — SVHN
**CSE 5526: Intro to Neural Networks — Spring 2026**

> **Instructions:** Only edit sections marked with `# TODO`. Do not modify any other code.  
> Run all cells before submitting. Unexecuted notebooks receive a zero.  
> Submit this single file as **`pa2.ipynb`** to your `pa2` folder on GitHub.


## 1. Imports and Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import time

# TODO: Set random seeds for reproducibility using seed 5526
# YOUR CODE HERE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')


## 2. Data Loading

Load the SVHN dataset. SVHN is already divided into `train` and `test` splits by torchvision — no manual splitting is needed.

> **Note:** SVHN images are 32×32 RGB. Labels are 0–9, where label `0` = digit `1`, ..., label `9` = digit `0`.

**Requirements:**
- Use `torchvision.datasets.SVHN` with `split='train'` and `split='test'`
- Apply `transforms.ToTensor()` only — no normalization
- Use a mini-batch size of **50**


In [ ]:
BATCH_SIZE = 50
CLASS_NAMES = ['1','2','3','4','5','6','7','8','9','0']

# TODO: Load SVHN train and test sets using torchvision.datasets.SVHN
# Apply transforms.ToTensor() only — no normalization
# Use split='train' for training data and split='test' for final evaluation
# YOUR CODE HERE


# TODO: Split the training set into 80% train and 20% validation
# Use random_split with a Generator seeded at 5526
# YOUR CODE HERE


# TODO: Create train_loader, val_loader, and test_loader using DataLoader
# Use BATCH_SIZE=50
# shuffle=True for train_loader, shuffle=False for val_loader and test_loader
# YOUR CODE HERE


print(f'Training samples:   {len(train_set)}')
print(f'Validation samples: {len(val_set)}')
print(f'Test samples:       {len(test_set)}')


## 3. Dataset Visualization

Display sample images to verify the dataset loaded correctly.

**Requirements:**
- Show at least **3 examples per digit class** (10 classes total)
- Label each row with the digit class


In [ ]:
# TODO: Visualize at least 3 sample images per digit class
# Display in a grid with digit labels on the left side
# YOUR CODE HERE


## 4. Model Definitions

### Xavier Initialization

Implement a helper function that applies Xavier uniform initialization to all `Conv2d` and `Linear` layers in a model, and sets all biases to zero.


In [ ]:
def xavier_init(model):
    """
    Apply Xavier uniform initialization to all Conv2d and Linear layers.
    Set all biases to zero.

    Args:
        model (nn.Module): the model to initialize in-place
    """
    # TODO
    pass


### Part I — Baseline CNN

Architecture:
- **Conv1:** 3 → 6 filters, 3×3 kernel, stride=1, padding=1 → ReLU
- **Conv2:** 6 → 10 filters, 3×3 kernel, stride=1, padding=1 → ReLU
- **MaxPool:** MaxPool2d(2, 2) at the end of the conv stack
- **FC:** 10×16×16 = 2560 → 10

Spatial flow: 32 → (conv×2) → MaxPool(2,2) → 16  


In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: Define self.conv_layers (nn.Sequential)
        self.conv_layers = None  # TODO

        # TODO: Define self.fc (nn.Linear, input=10*16*16=2560, output=10)
        self.fc = None  # TODO

    def forward(self, x):
        """
        Args:
            x (Tensor): input batch of shape (N, 3, 32, 32)
        Returns:
            Tensor: logits of shape (N, 10)
        """
        # TODO
        pass


baseline_model = BaselineCNN()
xavier_init(baseline_model)
print(f'Baseline parameters: {sum(p.numel() for p in baseline_model.parameters() if p.requires_grad):,}')
print(baseline_model)


### Part II-A — Pooling CNN

Same as baseline but with **MaxPool2d(3, stride=3) after each conv layer**.

Spatial flow: 32 → Conv→ReLU→MaxPool(3,s=3) → 10 → Conv→ReLU→MaxPool(3,s=3) → 3  
FC input: 10×3×3 = 90


In [ ]:
class PoolingCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: Define self.conv_layers with MaxPool2d(3, stride=3) after each conv+ReLU
        self.conv_layers = None  # TODO

        # TODO: Define self.fc (input=10*3*3=90, output=10)
        self.fc = None  # TODO

    def forward(self, x):
        """
        Args:
            x (Tensor): input batch of shape (N, 3, 32, 32)
        Returns:
            Tensor: logits of shape (N, 10)
        """
        # TODO
        pass


pooling_model = PoolingCNN()
xavier_init(pooling_model)
print(f'Pooling parameters: {sum(p.numel() for p in pooling_model.parameters() if p.requires_grad):,}')
print(pooling_model)


### Part II-B — Dropout CNN

Same as baseline but with **Dropout2d(p=0.25) after each conv+ReLU**, before the final MaxPool.

FC input: 10×16×16 = 2560 (spatial unchanged)


In [ ]:
class DropoutCNN(nn.Module):
    def __init__(self, dropout_rate=0.25):
        super().__init__()
        # TODO: Define self.conv_layers with Dropout2d(p=dropout_rate) after each conv+ReLU
        self.conv_layers = None  # TODO

        # TODO: Define self.fc (input=10*16*16=2560, output=10)
        self.fc = None  # TODO

    def forward(self, x):
        """
        Args:
            x (Tensor): input batch of shape (N, 3, 32, 32)
        Returns:
            Tensor: logits of shape (N, 10)
        """
        # TODO
        pass


dropout_model = DropoutCNN(dropout_rate=0.25)
xavier_init(dropout_model)
print(f'Dropout parameters: {sum(p.numel() for p in dropout_model.parameters() if p.requires_grad):,}')
print(dropout_model)


## 5. Training and Evaluation Functions

Implement all functions below. Function signatures, docstrings, and the epoch print statement are fixed — do not modify them.

**Training configuration (do not change):**
- Optimizer: SGD, lr=0.05, momentum=0.9, weight_decay=1e-4
- LR scheduler: StepLR, step_size=10, gamma=0.9
- Loss: CrossEntropyLoss
- Early stopping patience: 10 epochs
- Max epochs: 100


In [ ]:
def train_model(model, train_loader, val_loader, num_epochs=100,
                lr=0.05, momentum=0.9, patience=10):
    """
    Train model with SGD + StepLR + early stopping on validation loss.
    The test set is never used here — it is reserved for final evaluation only.

    Args:
        model        (nn.Module):   model to train
        train_loader (DataLoader):  training data
        val_loader   (DataLoader):  validation data (for early stopping)
        num_epochs   (int):         maximum number of epochs
        lr           (float):       initial learning rate
        momentum     (float):       SGD momentum
        patience     (int):         early stopping patience on val loss

    Returns:
        train_losses (list[float]): per-epoch training loss
        val_losses   (list[float]): per-epoch validation loss
        train_accs   (list[float]): per-epoch training accuracy
        val_accs     (list[float]): per-epoch validation accuracy
    """
    # TODO
    pass


In [ ]:
def evaluate_model(model, test_loader):
    """
    Evaluate model accuracy on test_loader.

    Args:
        model       (nn.Module):   trained model
        test_loader (DataLoader):  test data

    Returns:
        accuracy  (float):      fraction of correctly classified samples
        preds     (np.ndarray): predicted class indices, shape (N,)
        true_labels (np.ndarray): ground truth class indices, shape (N,)
    """
    # TODO
    pass


In [ ]:
def plot_learning_curves(train_losses, val_losses, train_accs, val_accs, title=''):
    """
    Plot loss and accuracy curves for train and validation sets side by side.

    Args:
        train_losses (list[float]): per-epoch training loss
        val_losses   (list[float]): per-epoch validation loss
        train_accs   (list[float]): per-epoch training accuracy
        val_accs     (list[float]): per-epoch validation accuracy
        title        (str):         overall figure title
    """
    # TODO: Two subplots — left: loss curves, right: accuracy curves
    # Include legend, axis labels, grid
    pass


In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names, title=''):
    """
    Display a labeled confusion matrix.

    Args:
        y_true      (np.ndarray): ground truth labels, shape (N,)
        y_pred      (np.ndarray): predicted labels, shape (N,)
        class_names (list[str]):  label names for axes
        title       (str):        figure title
    """
    # TODO: Use ConfusionMatrixDisplay, cmap='Blues', rotate x-tick labels 45 degrees
    pass


## 6. Part I — Baseline

In [ ]:
print('Training Baseline CNN...')
print('=' * 70)
train_losses, val_losses, train_accs, val_accs = train_model(
    baseline_model, train_loader, val_loader,
    num_epochs=100, lr=0.05, momentum=0.9, patience=10
)


### (1) Learning Curves

> **TODO (written answer):** Does the baseline overfit, underfit, or reasonably fit? Justify using the curves.


In [ ]:
plot_learning_curves(train_losses, val_losses, train_accs, val_accs,
                     title='Baseline CNN — SVHN Learning Curves')
print(f'Final Train Acc: {train_accs[-1]:.4f} | Final Val Acc: {val_accs[-1]:.4f}')



### (2) Test Accuracy and Confusion Matrix

In [ ]:
test_acc, preds, true_labels = evaluate_model(baseline_model, test_loader)
print(f'Baseline Val Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)')

plot_confusion_matrix(true_labels, preds, CLASS_NAMES,
                      title='Baseline CNN — Confusion Matrix')


---
## 7. Part II — Experiment 1: Max Pooling


In [ ]:
print('Training CNN + MaxPool(3x3, stride=3)...')
print('=' * 70)
p_train_losses, p_val_losses, p_train_accs, p_val_accs = train_model(
    pooling_model, train_loader, val_loader,
    num_epochs=100, lr=0.05, momentum=0.9, patience=10
)


### Pooling — Outcomes

> **TODO (written answer):** How does max pooling affect? Do the kernels look different?


In [ ]:
plot_learning_curves(p_train_losses, p_val_losses, p_train_accs, p_val_accs,
                     title='CNN + MaxPool(3x3, s=3) — SVHN Learning Curves')

test_acc_p, preds_p, labels_p = evaluate_model(pooling_model, test_loader)
print(f'Pooling Val Accuracy: {test_acc_p:.4f} ({test_acc_p*100:.2f}%)')

plot_confusion_matrix(labels_p, preds_p, CLASS_NAMES,
                      title='CNN + MaxPool(3x3, s=3) — Confusion Matrix')


---
## 8. Part II — Experiment 2: Dropout


In [ ]:
print('Training CNN + Dropout(p=0.25)...')
print('=' * 70)
d_train_losses, d_val_losses, d_train_accs, d_val_accs = train_model(
    dropout_model, train_loader, val_loader,
    num_epochs=100, lr=0.05, momentum=0.9, patience=10
)


### Dropout — Outcomes

> **TODO (written answer):** How does dropout affect? Compare the train/test loss gap to the baseline.


In [ ]:
plot_learning_curves(d_train_losses, d_val_losses, d_train_accs, d_val_accs,
                     title='CNN + Dropout(p=0.25) — SVHN Learning Curves')

test_acc_d, preds_d, labels_d = evaluate_model(dropout_model, test_loader)
print(f'Dropout Val Accuracy: {test_acc_d:.4f} ({test_acc_d*100:.2f}%)')

plot_confusion_matrix(labels_d, preds_d, CLASS_NAMES,
                      title='CNN + Dropout(p=0.25) — Confusion Matrix')

---
## 9. Part II — Experiment 3: Momentum Comparison (β=0.9 vs β=0.5)

Train the **same baseline architecture** with momentum β=0.5. All other settings are identical.  
Then compare against the baseline (β=0.9) results from Section 6.


In [ ]:
# TODO: Instantiate a new BaselineCNN, apply xavier_init, and train with momentum=0.5
# Store results as: m05_train_losses, m05_val_losses, m05_train_accs, m05_val_accs
# YOUR CODE HERE


### Momentum — Outcomes

> **TODO (written answer):** What does this tell you about the role of momentum in SGD?


In [ ]:
# TODO: Plot both runs (β=0.9 and β=0.5) on the same axes
# Two subplots: loss and accuracy. Distinguish runs by color or linestyle.
# YOUR CODE HERE


# TODO: Evaluate β=0.5 model and print test accuracy
# YOUR CODE HERE

print(f'Momentum β=0.9 Val Accuracy: {test_acc*100:.2f}%')
# print(f'Momentum β=0.5 Val Accuracy: ...')  # TODO: fill in

# TODO: Plot confusion matrix
# YOUR CODE HERE

---
## 10. Summary Table

In [ ]:
# TODO: Print a summary table of test accuracy across all four experiments
# Baseline (β=0.9) | MaxPooling | Dropout | Baseline (β=0.5)
# YOUR CODE HERE
